# MDP/DQN 호환성 분석 — Sequence 모델의 한계

SmartPitch MDP에서 전이확률 모델을 사용할 때의 실용적 제약 분석.

**핵심 질문**: 왜 Transformer/RNN은 MDP/DQN에 바로 적용하기 어려운가?

작성일: 2026-05-23

## 1. SmartPitch MDP 구조

```
State s_t = (balls, strikes, outs, base_runners, inning, batter_id, ...)
Action a_t = pitch_type (Fastball, Slider, Curveball, ...)
Reward r_t = outcome_value (Ball=-0.1, Strike=+0.1, HR=-2.0, Out=+1.0, ...)
Transition P(s_{t+1} | s_t, a_t) = f(state, action) → pitch outcome probs
```

### MDP State에서 복원 가능한 정보
- ✅ 현재 count (balls, strikes), outs, inning
- ✅ 타자 ID, 투수 ID
- ✅ 주자 상황
- ❌ **직전 400 pitch의 전체 시퀀스** ← Sequence 모델이 요구하는 것

Markov assumption: `P(s_{t+1} | s_t, a_t)` — 현재 state만으로 전이확률 결정.
MDP state는 요약 통계이지, raw pitch history가 아니다.

## 2. Sequence 모델의 MDP 호환 문제

### 2.1 입력 요구사항

| 모델 | 입력 | MDP state에서 복원? |
|------|------|--------------------|
| LR / LightGBM / MLP | 77-dim 단일 pitch | ✅ state에서 바로 생성 가능 |
| RNN (LSTM) | 400 pitches × 87-dim sequence | ❌ sequence history 필요 |
| Transformer | 400 pitches × 87-dim sequence | ❌ sequence history 필요 |

### 2.2 추론 속도

| 모델 | 추론 시간 (batch=1) | DQN lookahead 가능? |
|------|-------------------|--------------------|
| LR | <1ms | ✅ |
| LightGBM | ~1ms | ✅ |
| MLP | ~1ms (GPU) | ✅ |
| RNN | ~50-100ms (400 step) | ❌ (50-100× 느림) |
| Transformer | ~100-200ms (400 token) | ❌ (100-200× 느림) |

DQN은 매 step마다 Q(s,a) 계산 시 여러 action을 평가해야 함.
Transformer가 100ms라면 10 actions 평가 시 1초 — 실시간 의사결정 불가.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inference time comparison (log scale)
models = ['LR', 'LightGBM', 'MLP', 'RNN', 'Transformer']
times_ms = [0.1, 1, 1, 75, 150]  # estimated
colors = ['#3498db','#3498db','#3498db','#e74c3c','#e74c3c']
axes[0].bar(models, times_ms, color=colors, alpha=0.85)
axes[0].set_yscale('log')
axes[0].set_title('추론 시간 비교 (log scale, batch=1)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inference time (ms)')
axes[0].axhline(10, color='green', linestyle='--', alpha=0.7, label='10ms threshold (real-time)')
axes[0].legend()
for i, v in enumerate(times_ms):
    axes[0].text(i, v * 1.5, f'{v}ms', ha='center', fontsize=9)

# Accuracy vs MDP compatibility
acc = [0.411, 0.411, 0.411, 0.67, 0.672]  # 10-class top-1 (RNN estimated)
mdp_ok = [True, True, True, False, False]
colors2 = ['#3498db' if ok else '#e74c3c' for ok in mdp_ok]
axes[1].scatter(times_ms, acc, s=200, c=colors2, alpha=0.9, zorder=5)
for i, (t, a, m) in enumerate(zip(times_ms, acc, models)):
    axes[1].annotate(m, (t, a), textcoords='offset points', xytext=(8, 0), fontsize=10)
axes[1].set_xscale('log')
axes[1].set_title('정확도 vs 추론 속도 (MDP 호환성)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Inference time (ms, log)')
axes[1].set_ylabel('10-class Top-1 Accuracy')
axes[1].set_ylim(0.3, 0.85)
axes[1].axvline(10, color='green', linestyle='--', alpha=0.7)
blue_p = mpatches.Patch(color='#3498db', label='MDP 호환 ✅')
red_p  = mpatches.Patch(color='#e74c3c', label='MDP 비호환 ❌')
axes[1].legend(handles=[blue_p, red_p])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Hybrid 시스템 제안

```
┌─────────────────────────────────────────────────────────────┐
│                    Hybrid Architecture                       │
│                                                              │
│  [경기 전 / 타석 간]           [투구 직전]                   │
│  Transformer Analysis         MLP Decision                   │
│  400 pitches → embeddings     77-dim state → pitch probs     │
│  타자 패턴 파악               MDP/DQN action 선택            │
│  → feature augmentation       → 실시간 추론 (<1ms)           │
└─────────────────────────────────────────────────────────────┘
```

### 구현 방향
1. **Pre-game**: Transformer로 당일 타자의 pitch embedding 계산 (캐싱)
2. **In-game**: MLP(77-dim + embedding augment)로 실시간 pitch selection
3. **Advantage**: Transformer의 sequence 이해력 + MLP의 실시간성

## 4. 결론

| 시나리오 | 권장 모델 | 이유 |
|---------|---------|------|
| 학술 연구 | Transformer | 최고 정확도 67.2% |
| Production MDP | MLP (Model B) | 빠름, 호환성 |
| Hybrid Production | Transformer embedding + MLP | 최선의 trade-off |

### 미래 작업
- Small Transformer (12-layer → 2-layer): 파라미터 축소로 추론 가속
- Transformer embedding을 MDP state에 포함 (batter tendency feature)
- ONNX export로 C++/Java 추론 (서버 사이드)